# 03. QUBO 생성과 검증

확인 항목:
1. iteration 별 QUBO 생성
2. variable-role table
3. coefficient 통계
4. logical variables / edges / density
5. continuous MP, encoded MP, penalized QUBO 비교
6. 작은 QUBO exhaustive validation

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from cflp_bd_qa_ex2.config.loader import load_config
from cflp_bd_qa_ex2.data.io import load_all_instances

cfg = load_config(ROOT / "config" / "experiments" / "pilot.yaml")
instances = load_all_instances(ROOT / "data" / "raw")
by_id = {i.instance_id: i for i in instances}
print("configuration_hash:", cfg["configuration_hash"][:16])


configuration_hash: 0dc9f18b09b09302


## 1. Controlled trajectory 재구성

Gurobi-Benders 가 만든 cut trajectory 를 그대로 사용한다.
iteration `k` 의 QUBO 는 cut `0..k-1` 을 갖는다 (iteration 0 은 cut-free initial MP).

In [2]:
from cflp_bd_qa_ex2.benders.cuts import build_optimality_cut
from cflp_bd_qa_ex2.benders.solver import GurobiMasterProvider, run_benders

def cut_snapshots(inst):
    """iteration 별 cut 집합 목록을 만든다."""
    trace = run_benders(inst, GurobiMasterProvider(), cfg)
    cuts, snapshots = [], [[]]
    # **실제로 MP 에 추가된 cut 만** 사용한다.
    # 수렴 iteration 에서 생성되었으나 활성화되지 않은 마지막 cut 을 포함하면
    # 존재한 적 없는 MP 상태를 측정하게 된다.
    for snapshot in trace.cut_snapshots:
        if not snapshot.get("cut_activated", False):
            continue
        cuts = cuts + [build_optimality_cut(
            index=snapshot["cut_index"],
            capacity=inst.capacity.astype(float),
            u=np.asarray(snapshot["cut_dual_u"], dtype=float),
            v=np.asarray(snapshot["cut_dual_v"], dtype=float),
            iteration=snapshot["benders_iteration"],
            dual_minimality_status=snapshot["dual_minimality_status"],
        )]
        snapshots.append(list(cuts))
    return snapshots

target_ids = ["4x12_s100", "8x25_s200", "16x50_s300"]
snapshots = {tid: cut_snapshots(by_id[tid]) for tid in target_ids if tid in by_id}
{tid: len(v) for tid, v in snapshots.items()}


{'4x12_s100': 3, '8x25_s200': 8, '16x50_s300': 5}

## 2. Iteration 별 QUBO metric

coefficient range 는 **차이가 아니라 비율**이다.
`R_Q = max|Q_ab| / min|Q_ab|` (0이 아닌 계수 기준).

In [3]:
from cflp_bd_qa_ex2.qubo.builder import build_master_qubo
from cflp_bd_qa_ex2.qubo.diagnostics import (
    logical_graph, qubo_metrics, role_counts, source_graph_fingerprint,
)

rows = []
qubos = {}
for tid, snaps in snapshots.items():
    for iteration, cuts in enumerate(snaps):
        qubo = build_master_qubo(by_id[tid], cuts, cfg)
        qubos[(tid, iteration)] = qubo
        metrics = qubo_metrics(qubo)
        metrics.update(role_counts(qubo))
        metrics.update(qubo.constraint_metadata)
        metrics["source_graph_fingerprint"] = source_graph_fingerprint(logical_graph(qubo))
        rows.append({"instance_id": tid, "benders_iteration": iteration,
                     "num_cuts": len(cuts), **metrics})

qubo_frame = pd.DataFrame(rows)
qubo_frame[["instance_id", "benders_iteration", "num_cuts", "logical_variables",
            "logical_edges", "qubo_density", "qubo_log10_coefficient_range",
            "num_theta_bits", "num_capacity_slack_bits", "cut_slack_bits_total"]]


,instance_id,benders_iteration,num_cuts,logical_variables,logical_edges,qubo_density,qubo_log10_coefficient_range,num_theta_bits,num_capacity_slack_bits,cut_slack_bits_total
0,4x12_s100,0,0,22,66,0.285714,8.243367,10,8,0
1,4x12_s100,1,1,34,363,0.647059,7.334787,10,8,12
2,4x12_s100,2,2,46,585,0.565217,7.334787,10,8,24
3,8x25_s200,0,0,27,136,0.387464,8.757050,10,9,0
4,8x25_s200,1,1,39,521,0.703104,8.453751,10,9,12
5,8x25_s200,2,2,51,767,0.601569,8.453751,10,9,24
6,8x25_s200,3,3,63,1037,0.530978,8.629842,10,9,36
7,8x25_s200,4,4,75,1305,0.470270,8.754781,10,9,48
8,8x25_s200,5,5,87,1575,0.421010,8.851691,10,9,60
9,8x25_s200,6,6,99,1833,0.377860,8.930872,10,9,72


### cut slack bit 수 확인

정의는 `b_k = ceil(log2(S_k_max / delta_cut + 1))` 이다.

`B_k = 1 - (sum u - sum s v) / theta_U` 로 두면 `b_k = ceil(log2(4092 B_k + 1))` 이고,
`b_k = 12` 인 정확한 구간은 `0.500244 < B_k <= 1.000733` 이다.

아래에서 실제 `B_k` 를 계산해 이 구간 안에 있는지 본다.
**구간 안에 있다는 것은 관측 결과이지 이론적 보장이 아니다.**

In [4]:
bracket_rows = []
for tid, snaps in snapshots.items():
    inst = by_id[tid]
    theta_upper = inst.objective_upper_bound
    for cuts in snaps[-1:]:
        for cut in cuts:
            b_k = 1.0 - (cut.rhs_constant - float(cut.y_coefficients.sum())) / theta_upper
            bracket_rows.append({
                "instance_id": tid,
                "cut_index": cut.index,
                "B_k": b_k,
                "in_12bit_window": 0.500244 < b_k <= 1.000733,
            })

brackets = pd.DataFrame(bracket_rows)
print("B_k 범위: %.6f ~ %.6f" % (brackets["B_k"].min(), brackets["B_k"].max()))
print("12bit 구간 안:", brackets["in_12bit_window"].all())
brackets


B_k 범위: 0.935028 ~ 0.996131
12bit 구간 안: True


,instance_id,cut_index,B_k,in_12bit_window
0,4x12_s100,0,0.952092,True
1,4x12_s100,1,0.935028,True
2,8x25_s200,0,0.975874,True
3,8x25_s200,1,0.961282,True
4,8x25_s200,2,0.982993,True
5,8x25_s200,3,0.966323,True
6,8x25_s200,4,0.980310,True
7,8x25_s200,5,0.962522,True
8,8x25_s200,6,0.961904,True
9,16x50_s300,0,0.992033,True


## 3. Variable-role table

role 은 `y`, `theta_bit`, `capacity_slack_bit`, `cut_slack_bit` 네 종류이다.

In [5]:
from cflp_bd_qa_ex2.qubo.diagnostics import variable_role_table

key = ("4x12_s100", 1) if ("4x12_s100", 1) in qubos else list(qubos)[0]
role_table = pd.DataFrame(variable_role_table(qubos[key]))
print("QUBO:", key)
role_table


QUBO: ('4x12_s100', 1)


,variable,role,linear_coefficient,degree
0,y_0,y,-3.131123e+10,33
1,y_1,y,-4.683019e+10,33
2,y_2,y,-1.002392e+10,33
3,y_3,y,-2.249083e+09,11
4,theta_0,theta_bit,-8.165087e+08,24
5,theta_1,theta_bit,-1.628251e+09,24
6,theta_2,theta_bit,-3.237435e+09,24
7,theta_3,theta_bit,-6.398603e+09,24
8,theta_4,theta_bit,-1.249214e+10,24
9,theta_5,theta_bit,-2.376401e+10,24


## 4. 세 계층 비교: continuous MP / encoded MP / penalized QUBO

- `continuous_mp` : 유효한 lower bound 의 유일한 출처
- `encoded_mp`   : theta 만 binary encoding, 부등식 유지 -> **순수 discretization 영향**
- `penalized_qubo`: slack + penalty 포함 -> 실제 SA/QA 가 푸는 문제

In [6]:
from cflp_bd_qa_ex2.encoding.theta import build_theta_encoding
from cflp_bd_qa_ex2.evaluation.metrics import discretization_effect
from cflp_bd_qa_ex2.models.master import solve_continuous_mp, solve_encoded_mp
from cflp_bd_qa_ex2.qubo.exact import validate_qubo_exactly

# 주의: exact validation 은 **가장 비싼 단계**이다.
#   - 22 변수 exhaustive enumeration: 약 70초 (2^22 = 4.2M)
#   - MIQP 는 변수 수가 늘수록 급격히 느려지므로
#     validation.miqp_tractable_max_variables(기본 50)을 넘으면 건너뛴다
#   - validation.exact_time_limit(기본 60초) 안에 증명하지 못하면 TIME_LIMIT
# 아래 설정으로 실제 소요 시간을 조절한다.
print("exhaustive <= %d vars | MIQP <= %d vars | time limit %.0fs"
      % (cfg["validation"]["exhaustive_max_variables"],
         cfg["validation"]["miqp_tractable_max_variables"],
         cfg["validation"]["exact_time_limit"]))

layer_rows = []
for (tid, iteration), qubo in sorted(qubos.items()):
    inst = by_id[tid]
    cuts = snapshots[tid][iteration]

    continuous = solve_continuous_mp(inst, cuts, cfg)
    theta_encoding = build_theta_encoding(inst.objective_upper_bound,
                                          int(cfg["encoding"]["theta"]["bits"]))
    encoded = solve_encoded_mp(inst, cuts, theta_encoding, cfg)

    started = time.perf_counter()
    exact = validate_qubo_exactly(qubo, inst, cuts, cfg)
    elapsed = time.perf_counter() - started
    print(f"  {tid} it{iteration} n={qubo.num_variables:3d} "
          f"{elapsed:6.1f}s {exact.status} ({exact.method})")

    layer_rows.append({
        "instance_id": tid,
        "benders_iteration": iteration,
        "logical_variables": qubo.num_variables,
        "continuous_mp_objective": continuous.objective,
        "encoded_mp_objective": encoded.objective,
        "qubo_ground_state_energy": exact.ground_state_energy,
        "exact_incumbent_energy": exact.fields.get("exact_incumbent_energy"),
        "exact_status": str(exact.status),
        "exact_method": exact.method,
        "exact_skip_reason": exact.fields.get("exact_skip_reason"),
        "exact_runtime": elapsed,
        "penalty_status": str(exact.penalty_status) if exact.penalty_status else None,
        **discretization_effect(continuous.objective, encoded.objective),
    })

layers = pd.DataFrame(layer_rows)
layers


exhaustive <= 25 vars | MIQP <= 50 vars | time limit 60s
  16x50_s300 it0 n= 36    1.2s EXACT_QUBO_VALIDATED (gurobi_miqp)
  16x50_s300 it1 n= 48   60.2s TIME_LIMIT (gurobi_miqp)
  16x50_s300 it2 n= 60    0.0s EXACT_QUBO_SKIPPED_TOO_LARGE (gurobi_miqp)
  16x50_s300 it3 n= 72    0.0s EXACT_QUBO_SKIPPED_TOO_LARGE (gurobi_miqp)
  16x50_s300 it4 n= 84    0.0s EXACT_QUBO_SKIPPED_TOO_LARGE (gurobi_miqp)
  4x12_s100 it0 n= 22   21.2s EXACT_QUBO_VALIDATED (exhaustive)
  4x12_s100 it1 n= 34    2.7s EXACT_QUBO_VALIDATED (gurobi_miqp)
  4x12_s100 it2 n= 46   36.9s EXACT_QUBO_VALIDATED (gurobi_miqp)
  8x25_s200 it0 n= 27    0.1s EXACT_QUBO_VALIDATED (gurobi_miqp)
  8x25_s200 it1 n= 39   32.1s EXACT_QUBO_VALIDATED (gurobi_miqp)
  8x25_s200 it2 n= 51    0.0s EXACT_QUBO_SKIPPED_TOO_LARGE (gurobi_miqp)
  8x25_s200 it3 n= 63    0.0s EXACT_QUBO_SKIPPED_TOO_LARGE (gurobi_miqp)
  8x25_s200 it4 n= 75    0.0s EXACT_QUBO_SKIPPED_TOO_LARGE (gurobi_miqp)
  8x25_s200 it5 n= 87    0.0s EXACT_QUBO_SKIPPED_TOO_LAR

,instance_id,benders_iteration,logical_variables,continuous_mp_objective,encoded_mp_objective,qubo_ground_state_energy,exact_incumbent_energy,exact_status,exact_method,exact_skip_reason,exact_runtime,penalty_status,discretization_absolute,discretization_relative
0,16x50_s300,0,36,12744.290618,12744.290618,12744.290649,NaN,EXACT_QUBO_VALIDATED,gurobi_miqp,NaN,1.224946,SUCCESS,0.000000,0.000000
1,16x50_s300,1,48,15964.101520,15975.180235,NaN,33293.488281,TIME_LIMIT,gurobi_miqp,NaN,60.151192,NaN,11.078716,0.000694
2,16x50_s300,2,60,16447.304238,16501.029915,NaN,NaN,EXACT_QUBO_SKIPPED_TOO_LARGE,gurobi_miqp,intractable,0.000019,NaN,53.725676,0.003267
3,16x50_s300,3,72,16923.257037,16978.398020,NaN,NaN,EXACT_QUBO_SKIPPED_TOO_LARGE,gurobi_miqp,intractable,0.000018,NaN,55.140983,0.003258
4,16x50_s300,4,84,16931.156496,17003.910579,NaN,NaN,EXACT_QUBO_SKIPPED_TOO_LARGE,gurobi_miqp,intractable,0.000023,NaN,72.754083,0.004297
5,4x12_s100,0,22,3223.213313,3223.213313,3223.213314,NaN,EXACT_QUBO_VALIDATED,exhaustive,NaN,21.228648,SUCCESS,0.000000,0.000000
6,4x12_s100,1,34,4815.644243,4817.031299,6057.299255,NaN,EXACT_QUBO_VALIDATED,gurobi_miqp,NaN,2.682219,PENALTY_INSUFFICIENT,1.387055,0.000288
7,4x12_s100,2,46,4904.433098,4905.538086,9044.330170,NaN,EXACT_QUBO_VALIDATED,gurobi_miqp,NaN,36.859566,SUCCESS,1.104988,0.000225
8,8x25_s200,0,27,6911.147382,6911.147382,6911.147392,NaN,EXACT_QUBO_VALIDATED,gurobi_miqp,NaN,0.080707,SUCCESS,0.000000,0.000000
9,8x25_s200,1,39,8898.287135,8931.647510,9252.714111,NaN,EXACT_QUBO_VALIDATED,gurobi_miqp,NaN,32.055092,SUCCESS,33.360375,0.003749


## 5. Exhaustive validation 의 실제 적용 범위

threshold 25 기준으로 열거 가능한 범위는 **4x12 의 cut-free initial MP (22 변수)** 뿐이다.

- cut 이 하나 추가되면 34 변수 -> 불가능
- 8x25 초기 MP 는 27 변수 -> 이미 초과
- 16x50 초기 MP 는 36 변수 -> 이미 초과

초과 시 임의로 축소하지 않고 `EXACT_QUBO_SKIPPED_TOO_LARGE` 로 기록한다.
Gurobi MIQP 는 제한 라이선스에서 200 변수까지만 가능하다.

In [7]:
scope = layers.groupby("exact_status").size().rename("count")
print(scope)
print()
print("변수 수 기준:")
print(qubo_frame[["instance_id", "benders_iteration", "logical_variables"]].to_string(index=False))


exact_status
EXACT_QUBO_SKIPPED_TOO_LARGE    9
EXACT_QUBO_VALIDATED            6
TIME_LIMIT                      1
Name: count, dtype: int64

변수 수 기준:
instance_id  benders_iteration  logical_variables
  4x12_s100                  0                 22
  4x12_s100                  1                 34
  4x12_s100                  2                 46
  8x25_s200                  0                 27
  8x25_s200                  1                 39
  8x25_s200                  2                 51
  8x25_s200                  3                 63
  8x25_s200                  4                 75
  8x25_s200                  5                 87
  8x25_s200                  6                 99
  8x25_s200                  7                111
 16x50_s300                  0                 36
 16x50_s300                  1                 48
 16x50_s300                  2                 60
 16x50_s300                  3                 72
 16x50_s300                  4                 84

## 6. QUBO energy 검증

직접 계산한 energy 와 dimod BQM energy 가 일치해야 한다.

In [8]:
import random

rng = random.Random(0)
max_error = 0.0
for qubo in qubos.values():
    bqm = qubo.to_bqm()
    for _ in range(20):
        assignment = {v: rng.randint(0, 1) for v in qubo.variable_labels}
        direct, reference = qubo.energy(assignment), bqm.energy(assignment)
        max_error = max(max_error, abs(direct - reference) / max(1.0, abs(reference)))

print("최대 상대 오차: %.3e" % max_error)
assert max_error < 1e-9


최대 상대 오차: 4.826e-13


## 해석

- iteration 당 변수 증가량이 `theta_bits + log2(1/r_cut) = 12` 근처이면,
  증가 속도가 **인스턴스 크기가 아니라 theta 해상도에 지배된다**는 뜻이다.
  단 이것은 관측이며 `B_k` 가 12bit 구간을 벗어나면 깨진다.
- `qubo_log10_coefficient_range` 가 8 을 넘으면 QA 에 매우 불리하다.
  하드웨어가 제공하는 유효 정밀도는 이보다 훨씬 낮기 때문이다.
- `continuous_mp` 와 `encoded_mp` 의 차이가 곧 theta discretization 손실이다.
  penalized QUBO 와의 추가 차이는 slack/penalty 때문이다.
- exhaustive validation 이 사실상 4x12 초기 MP 에만 적용된다는 점은
  본 파일럿의 명확한 한계이다.